# Batch vs stream scoring for real-time fraud detection

This notebook compares two ways of scoring the same fraud model in Apache PySpark:

- **Batch:** the whole test set is scored in one Spark job.
- **Stream simulation:** each test record is scored in its own Spark job, which mimics a transaction arriving alone.

Both pipelines use the same trained Random Forest, the same features and the same split. Only the scoring pattern changes.

The notebook writes its outputs to `results/`. Set `TEST_LIMIT` below to a small number for a quick check. Leave it as `None` for the full experiment. The full stream run takes roughly 2.5 hours on a Colab CPU runtime.

In [ ]:
%pip install -q pyspark faker pandas numpy scikit-learn psutil matplotlib

## 1. Configuration

In [ ]:
import os
import random
import time

import numpy as np
import pandas as pd
import psutil

DATA_DIR = os.environ.get("DATA_DIR", "data")          # on Colab set to "/content/drive/MyDrive/fraud_exp"
RESULTS_DIR = os.environ.get("RESULTS_DIR", "results")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
DATA_PATH = os.path.join(DATA_DIR, "transactions.csv")

NUM_RECORDS = 100_000
FRAUD_RATE = 0.03
SEED = 42

# None = score the full test set. Set an integer for a smoke test.
TEST_LIMIT = int(os.environ["TEST_LIMIT"]) if os.environ.get("TEST_LIMIT") else None

FEATURES = ["amount", "hour_of_day", "is_night", "amount_zscore", "merchant_idx", "account_age_days"]

## 2. Generate the synthetic dataset

The generator keeps the draw order from the version used in the dissertation, so the fraud count should be 3,063 for seed 42.
Faker is seeded here. The earlier run did not seed it. This only changes the `location` column, which the model does not use.

In [ ]:
from datetime import datetime
from faker import Faker

ALL_CATEGORIES = [
    "GROCERIES", "FUEL", "RESTAURANT", "RETAIL",
    "UTILITIES", "ENTERTAINMENT", "HEALTHCARE",
    "FOREIGN_TRANSFER", "CRYPTO", "WIRE_TRANSFER",
]

def generate_dataset(path):
    fake = Faker()
    fake.seed_instance(SEED)
    random.seed(SEED)
    np.random.seed(SEED)
    records = []
    for i in range(NUM_RECORDS):
        is_fraud = 1 if random.random() < FRAUD_RATE else 0
        if is_fraud:
            # Fraud is likely to have these traits, but not guaranteed.
            amount = round(random.uniform(200, 50000), 2)
            hour = random.choices(range(24), weights=[
                1, 3, 5, 5, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2
            ])[0]
            merchant_category = random.choices(ALL_CATEGORIES, weights=[3, 3, 3, 3, 3, 3, 3, 10, 10, 8])[0]
            account_age_days = random.randint(1, 3650)
        else:
            amount = round(random.uniform(5, 8000), 2)
            hour = random.randint(0, 23)
            merchant_category = random.choices(ALL_CATEGORIES, weights=[10, 10, 10, 10, 10, 10, 10, 2, 2, 2])[0]
            account_age_days = random.randint(30, 3650)
            # Label-independent amount noise.
            if random.random() < 0.05:
                amount = round(random.uniform(3000, 15000), 2)
        timestamp = datetime(
            2024,
            random.randint(1, 12),
            random.randint(1, 28),
            hour,
            random.randint(0, 59),
            random.randint(0, 59),
        )
        records.append({
            "transaction_id": f"TXN-{i + 1:06d}",
            "timestamp": timestamp,
            "customer_id": f"CUST-{random.randint(1000, 9999)}",
            "amount": amount,
            "merchant_category": merchant_category,
            "location": fake.city(),
            "account_age_days": account_age_days,
            "is_fraud": is_fraud,
        })
    df = pd.DataFrame(records)
    df.to_csv(path, index=False)
    return df

if not os.path.exists(DATA_PATH):
    df_raw = generate_dataset(DATA_PATH)
else:
    df_raw = pd.read_csv(DATA_PATH)

print(f"Records: {len(df_raw):,}")
print(f"Fraud:   {df_raw['is_fraud'].sum():,} ({df_raw['is_fraud'].mean() * 100:.2f}%)")
print(f"Mean amount, fraud: {df_raw.loc[df_raw.is_fraud == 1, 'amount'].mean():,.2f}")
print(f"Mean amount, legit: {df_raw.loc[df_raw.is_fraud == 0, 'amount'].mean():,.2f}")

## 3. Start Spark and prepare the data

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, DoubleType,
                               IntegerType, TimestampType)
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier

spark = (SparkSession.builder
         .appName("BatchVsStreamFraud")
         .config("spark.driver.memory", "4g")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

RAW_SCHEMA = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("customer_id", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("merchant_category", StringType(), True),
    StructField("location", StringType(), True),
    StructField("account_age_days", IntegerType(), True),
    StructField("is_fraud", IntegerType(), True),
])

raw = spark.read.csv(DATA_PATH, header=True, schema=RAW_SCHEMA)

# Global amount statistics, computed once and shared by both pipelines.
_s = raw.select(F.mean("amount").alias("m"), F.stddev("amount").alias("s")).collect()[0]
MEAN_AMT, STD_AMT = float(_s["m"]), float(_s["s"])

def add_features(df):
    """Feature engineering used by both pipelines. Each row depends only on itself and the constants above."""
    df = df.withColumn("hour_of_day", F.hour("timestamp"))
    df = df.withColumn("is_night",
                       F.when((F.col("hour_of_day") >= 1) & (F.col("hour_of_day") <= 4), 1).otherwise(0))
    df = df.withColumn("amount_zscore", (F.col("amount") - F.lit(MEAN_AMT)) / F.lit(STD_AMT))
    return df.fillna(0, subset=["hour_of_day", "is_night", "amount_zscore"])

train_raw, test_raw = raw.randomSplit([0.8, 0.2], seed=SEED)
if TEST_LIMIT:
    test_raw = test_raw.limit(TEST_LIMIT)
test_raw = test_raw.cache()
n_train = train_raw.count()
n_test = test_raw.count()
print(f"Train: {n_train:,}  Test: {n_test:,}")

## 4. Train the model once (shared by both pipelines)

Training time is recorded separately and is not part of either scoring measurement.

In [ ]:
indexer = StringIndexer(inputCol="merchant_category", outputCol="merchant_idx")
assembler = VectorAssembler(inputCols=FEATURES, outputCol="features")
rf = RandomForestClassifier(labelCol="is_fraud", featuresCol="features", numTrees=100, seed=SEED)
pipeline = Pipeline(stages=[indexer, assembler, rf])

t0 = time.perf_counter()
model = pipeline.fit(add_features(train_raw))
train_time = time.perf_counter() - t0
print(f"Training time: {train_time:.2f} s")

## 5. Batch pipeline

The timer covers feature engineering, prediction and collection of the results to Python. Nothing else is timed.

In [ ]:
process = psutil.Process(os.getpid())
psutil.cpu_percent(interval=None)            # first call primes the counter
mem_before = process.memory_info().rss / 1024**2

t0 = time.perf_counter()
batch_pred = (model.transform(add_features(test_raw))
              .select("transaction_id", "is_fraud", "prediction")
              .toPandas())
batch_time = time.perf_counter() - t0

batch_cpu = psutil.cpu_percent(interval=None)
mem_after = process.memory_info().rss / 1024**2
batch_mem = mem_after - mem_before

print(f"Batch: {len(batch_pred):,} records in {batch_time:.2f} s "
      f"({len(batch_pred) / batch_time:,.0f} records/s)")

## 6. Stream simulation

Each test record is turned into a one-row Spark DataFrame and scored alone. The timer covers feature engineering, prediction and collection, which matches the batch timer. Building the one-row DataFrame is outside the timer.

In [ ]:
test_pdf = test_raw.toPandas()
test_pdf["timestamp"] = pd.to_datetime(test_pdf["timestamp"])

latencies, stream_rows, cpu_samples = [], [], []
psutil.cpu_percent(interval=None)
mem_before = process.memory_info().rss / 1024**2

print(f"Scoring {len(test_pdf):,} records one at a time...")
for n, rec in enumerate(test_pdf.itertuples(index=False), start=1):
    one = spark.createDataFrame([(
        str(rec.transaction_id),
        rec.timestamp.to_pydatetime(),
        str(rec.customer_id),
        float(rec.amount),
        str(rec.merchant_category),
        str(rec.location),
        int(rec.account_age_days),
        int(rec.is_fraud),
    )], schema=RAW_SCHEMA)

    t0 = time.perf_counter()
    pred = model.transform(add_features(one)).select("prediction").collect()[0][0]
    latencies.append((time.perf_counter() - t0) * 1000)

    stream_rows.append((rec.transaction_id, int(rec.is_fraud), float(pred)))

    if n % 1000 == 0:
        cpu_samples.append(psutil.cpu_percent(interval=None))
        print(f"  {n:,} / {len(test_pdf):,}  mean latency so far: {np.mean(latencies):.2f} ms")

stream_time = sum(latencies) / 1000
stream_cpu = max(cpu_samples) if cpu_samples else psutil.cpu_percent(interval=None)
stream_mem = process.memory_info().rss / 1024**2 - mem_before
stream_pred = pd.DataFrame(stream_rows, columns=["transaction_id", "is_fraud", "prediction"])
print(f"Stream: {len(stream_pred):,} records in {stream_time:.2f} s")

## 7. Results

Two sets of classification metrics are shown on purpose:

- **Fraud class:** precision, recall and F1 for the fraud label only.
- **Class-weighted:** each class's score averaged by its share of the test set. This is what Spark's default evaluator reports. On a 3% fraud problem it is dominated by the legitimate class.

The earlier analysis compared class-weighted batch numbers with fraud-class stream numbers. Both sets are computed for both pipelines here, so the comparison is like for like.

In [ ]:
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, accuracy_score

def metrics_block(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    p_f, r_f, f_f, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", pos_label=1, zero_division=0)
    p_w, r_w, f_w, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    return {
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "accuracy": accuracy_score(y_true, y_pred),
        "fraud_precision": p_f, "fraud_recall": r_f, "fraud_f1": f_f,
        "weighted_precision": p_w, "weighted_recall": r_w, "weighted_f1": f_w,
    }

batch_m = metrics_block(batch_pred["is_fraud"], batch_pred["prediction"].astype(int))
stream_m = metrics_block(stream_pred["is_fraud"], stream_pred["prediction"].astype(int))

summary = pd.DataFrame({"batch": batch_m, "stream": stream_m})
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
print(summary)

In [ ]:
merged = batch_pred.merge(stream_pred, on="transaction_id", suffixes=("_batch", "_stream"))
assert len(merged) == len(batch_pred) == len(stream_pred), "transaction IDs do not line up"
disagree = merged[merged["prediction_batch"] != merged["prediction_stream"]]
print(f"Records scored: {len(merged):,}")
print(f"Records where batch and stream predictions differ: {len(disagree):,}")
disagree.head(20)

## 8. Timing and resources

In [ ]:
timing = pd.DataFrame([
    {"pipeline": "batch", "records": len(batch_pred), "elapsed_s": batch_time,
     "mean_latency_ms": batch_time * 1000 / len(batch_pred),
     "throughput_rec_s": len(batch_pred) / batch_time, "peak_cpu_pct": batch_cpu, "mem_increase_mb": batch_mem},
    {"pipeline": "stream", "records": len(stream_pred), "elapsed_s": stream_time,
     "mean_latency_ms": float(np.mean(latencies)),
     "throughput_rec_s": len(stream_pred) / stream_time, "peak_cpu_pct": stream_cpu, "mem_increase_mb": stream_mem},
])
timing["slowdown_vs_batch"] = timing["elapsed_s"] / timing.loc[0, "elapsed_s"]
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
print(f"Training time (not included above): {train_time:.2f} s")
timing

## 9. Save outputs

In [ ]:
timing.to_csv(os.path.join(RESULTS_DIR, "timing.csv"), index=False)
summary.to_csv(os.path.join(RESULTS_DIR, "classification_metrics.csv"))
batch_pred.to_csv(os.path.join(RESULTS_DIR, "batch_predictions.csv"), index=False)
stream_pred.to_csv(os.path.join(RESULTS_DIR, "stream_predictions.csv"), index=False)
print("Saved to", os.path.abspath(RESULTS_DIR))